## Embeddings + Vector Search (ChromaDB)

In [2]:
import sys
print(sys.executable)

/home/project/response-review-assistant/.venv/bin/python


In [3]:
import pandas as pd
import numpy as np

DATA_PATH = '../data/processed/reviews_clean.parquet'
CHROMA_PATH = '../chroma_db'
COLLECTION = 'amzaon_reviews'
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

df = pd.read_parquet(DATA_PATH)
print(df.shape)
df.head(2)

(20987, 5)


,document,rating,review_date,country,review_id
0,A Store That Doesn't Want to Sell Anything. I ...,1,2024-09-16 13:44:26+00:00,US,0
1,Had multiple orders one turned up and…. Had mu...,1,2024-09-16 18:26:46+00:00,GB,1


In [4]:
from sentence_transformers import SentenceTransformer,util

model = SentenceTransformer(MODEL_NAME)
print("dim:", model.get_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

dim: 384


In [5]:
q = "package arrived very late"
candidates = [
    "delivery took three weeks longer than promised",
    "the delivery driver was very polite",
    "I love the new phone I bought",
    ]
q_emb = model.encode(q)
for c in candidates:
    print(f"{util.cos_sim(q_emb, model.encode(c)).item():.3f} {c}")

0.535 delivery took three weeks longer than promised
0.446 the delivery driver was very polite
0.133 I love the new phone I bought


In [6]:
a = "Late delivery. The packaging was damaged and support never replied."
b = "Late delivery. Late delivery. The packaging was damaged and support never replied."
c = "Late delivery"
print("no-dup vs 'Late delivery':", round(util.cos_sim(model.encode(a),model.encode(c)).item(),3))
print("dup vs 'Late delivery':", round(util.cos_sim(model.encode(b),model.encode(c)).item(),3))

no-dup vs 'Late delivery': 0.569
dup vs 'Late delivery': 0.588


In [7]:
documents = df['document'].tolist()

embeddings = model.encode(
    documents,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)
print(embeddings.shape)

Batches:   0%|          | 0/328 [00:00<?, ?it/s]

(20987, 384)


In [8]:
import chromadb

client = chromadb.PersistentClient(path=CHROMA_PATH)

try:
    client.delete_collection(COLLECTION)
except Exception:
    pass

collection = client.create_collection(
    COLLECTION,
    metadata={"hnsw:space": "cosine"},
)

In [10]:
metadatas = [
    {
        "rating": int(r),
        "country": str(c),
        "review_date":str(d.date()),
    }
    for r,c,d in zip(df['rating'],df['country'],df['review_date'])
]
ids = df['review_id'].tolist()

BATCH = 5000
for i in range(0, len(ids), BATCH):
    collection.add(
        ids=ids[i:i+BATCH],
        embeddings=embeddings[i:i+BATCH].tolist(),
        documents=documents[i:i+BATCH],
        metadatas=metadatas[i:i+BATCH],
    )
    print(f"added {min(i+BATCH, len(ids)):,}/{len(ids):,}")

print("total in collection:", collection.count())

added 5,000/20,987
added 10,000/20,987
added 15,000/20,987
added 20,000/20,987
added 20,987/20,987
total in collection: 20987


In [11]:
def search(query,k=5,where=None):
    q_emb = model.encode(query, normalize_embeddings=True)
    res = collection.query(
        query_embeddings=[q_emb.tolist()],
        n_results=k,
        where=where,
    )
    print(f"Query: {query!r}" + (f" | filter: {where}" if where else""))
    print("=" * 80)

    for doc,meta,dist in zip(res['documents'][0], res['metadatas'][0], res['distances'][0]):
        print(f"[{meta['rating']}* {meta['country']} {meta['review_date']}] dist={dist:.3f}")
        print(f" {doc[:200]}{'...' if len(doc) > 200 else ''}\n")

search("account frozen asking for verification documents")

Query: 'account frozen asking for verification documents'
[1* US 2020-07-09] dist=0.376
 They have freeze my account over a…. They have freeze my account over a $20.00 dispute and I can't talk to anyone to fix it. They say it could take up to 30 days. Funny when I called 2 days ago they s...

[1* US 2019-12-28] dist=0.409
 Our account was frozen because we…worst customer service ever. Our account was frozen because we cancelled payment on items we thought fraudulent. After reviewing further, discovered that the purchase...

[1* RO 2019-10-17] dist=0.423
 Absolutely STUPID customer care and…. Absolutely STUPID customer care and password reset system.I called amazon support and asked the guy for a way to verify my account:I was logged in but the ret**rd...

[1* GB 2022-05-29] dist=0.424
 LIARS AND ACCOUNT ON HOLD. was given a faulty item, product page shows "FREE RETURNS" but when trying to return they tell me to pay for my own label.account is always getting temporarily placed on hol...

In [ ]:
search("refund never arrived after returning item")

Query: 'refund never arrived after returning item'
[1* GB 2022-09-28] dist=0.177
 I returned an item a month ago and I'm…. I returned an item a month ago and I'm yet to get any refund. Please advise

[1* GB 2022-08-23] dist=0.207
 Refund. Be careful when returning items just made me wait 3 weeks for a refund for an item they collected the following day.  Make rules up as they go along and almost impossible to contact very dissa...

[1* US 2023-05-01] dist=0.228
 I return item and didn’t get refund my…. I return item and didn’t get refund my money ,I called coustomer service they took More than half hour lady name wasDaniala ,bad coustomer service

[1* GB 2021-01-16] dist=0.244
 Item never arrived. Now I have to ask for the refund.

[5* US 2019-12-31] dist=0.263
 No Hassle When Returning Items. Amazon refunds your money quickly when returning items; oftentimes within a few hours



In [13]:
search("ordered something but it never showed up")

Query: 'ordered something but it never showed up'
[2* GB 2021-06-27] dist=0.321
 A little unreliable ordered something showed as being delayed then never showed up 4 weeks now. Someone needs to be doing something about it.

[2* GB 2022-05-09] dist=0.356
 I had an order that was suppose to come…. I had an order that was suppose to come to me last week I got one item not the other I called and confirmed on Saturday if it would be with me the gentleman s...

[3* GB 2019-01-04] dist=0.376
 FRUSTRATED. I have brought lots of things from Amazon and I do love the company, however I ordered something so exciting in about March time and it never ever turned up, I got an email saying they had...

[1* GB 2023-01-24] dist=0.381
 Ive ordered yesterday the 23rd and I…. Ive ordered yesterday the 23rd and I need the item on Wednesday and they sent me an email says item delivered and I never ever got my order yet and I tried to ra...

[1* US 2024-08-07] dist=0.391
 Check order haven't received



In [14]:
search("fast delivery good experience", k=5, where={"rating": {"$gte": 4}})

Query: 'fast delivery good experience' | filter: {'rating': {'$gte': 4}}
[5* BH 2024-06-28] dist=0.184
 Excellent experience, variety of items and fast delivery

[5* GB 2022-06-15] dist=0.190
 Fantastic experience - quick delivery & easy to deal with

[5* GB 2021-04-01] dist=0.206
 fast delivery, good deals. Super fast delivery. Amazing products (double check the description and reviews before buying)

[5* GB 2011-11-21] dist=0.213
 great. absolutely amazuing company fast convenient delivery

[5* GB 2012-01-24] dist=0.224
 Fast delivery and great company. This company always deliver when promised



In [15]:
# review แง่ลบจากประเทศเดียว
search("customer service was unhelpful", k=5,
       where={"$and": [{"rating": {"$lte": 2}}, {"country": "GB"}]})

Query: 'customer service was unhelpful' | filter: {'$and': [{'rating': {'$lte': 2}}, {'country': 'GB'}]}
[1* GB 2024-03-16] dist=0.301
 Absolutely horrible customer service

[1* GB 2019-12-14] dist=0.313
 Customer service team completely…. Customer service team completely disrespectful and unprofessional. Spoke to a woman named Jasmine P and received the most unfriendly and unwelcoming service and she ...

[1* GB 2022-01-26] dist=0.325
 A very rude unhelpful customer service…. A very rude unhelpful customer service we have your money attitude service is poor at best I give you 1 star couldn't give you zero

[1* GB 2024-02-04] dist=0.341
 Customer service is a disgrace. -speak to you like garbage, refuse to log a complaint and to state who they are regulated by - complete law to themselves- absolute garbage

[1* GB 2024-04-16] dist=0.348
 The customer service is appalling…. The customer service is appalling refuse to help and when you want to put forward about a complaint they refuse an